# 06 - Dashboard data export
Plain cricket counts for every player (overall, under pressure, by phase, by season) + category / nation / status.
Reads the newest data/player_eligibility*.csv. After retention / trade news: edit that CSV, rerun this notebook, reopen outputs/dashboard.html.

In [5]:
import json
import numpy as np, pandas as pd
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
DATA, OUT = ROOT / 'data', ROOT / 'outputs'
PRESSURE = 80

ELIG = sorted(DATA.glob('player_eligibility*.csv'), key=lambda p: p.stat().st_mtime)[-1]
E = pd.read_csv(ELIG, encoding='utf-8-sig')
E.columns = [c.strip().lower() for c in E.columns]
E['player_id'] = pd.to_numeric(E.player_id, errors='coerce')
E = E.dropna(subset=['player_id']).astype({'player_id': int})
for c in ['team', 'indian_or_overseas', 'capped_indian', 'current_ipl_team']:
    E[c] = E[c].astype(str).str.strip().str.lower().replace({'nan': np.nan, '': np.nan})
E['current_ipl_team'] = E.current_ipl_team.fillna('available')
oc = next((c for c in ('order', 'orders') if c in E.columns), None)         # usual batting position, written in the CSV
E['orders'] = E[oc].astype(str).str.strip().str.lower() if oc else 'na'
E['orders'] = E.orders.map({'top': 'Top (1–3)', 'middle': 'Middle (4–7)', 'lower': 'Lower (8–11)'}).fillna('Not specified')
E = E.drop_duplicates('player_id', keep='last').set_index('player_id')

Bt = pd.read_parquet(OUT / 'batter_ball_values.parquet')
Bw = pd.read_parquet(OUT / 'bowler_ball_values.parquet')
PAK = set(Bt[Bt.bat_team.str.contains('Pakistan')].striker_id) | set(Bw[Bw.bowl_team.str.contains('Pakistan')].bowler_id)
E = E[~E.index.isin(PAK) & E.current_ipl_team.ne('retired')]

# nation: the T20I side a player has played for (associates are not lumped together)
mp = pd.read_csv(DATA / 'competition_team_player_mapping.csv').merge(pd.read_csv(DATA / 'competitions.csv')[['comp_id', 'match_format']], on='comp_id')
nat = mp[mp.match_format == 'T20I'].groupby('player_id').team_name.agg(lambda s: s.value_counts().index[0])
nt = pd.Series(E.index.map(nat), index=E.index)
foreign = nt.notna() & nt.ne('India')            # played T20Is for a non-India side => overseas, whatever the CSV says
print('category overrides (CSV said Indian, T20I nation is not India): %d' % (foreign & E.indian_or_overseas.eq('indian')).sum())
E.loc[foreign, 'indian_or_overseas'] = 'overseas'
E['nation'] = nt.fillna(E.team.str.title())
E['category'] = np.select([E.indian_or_overseas.eq('overseas'), E.capped_indian.eq('yes')], ['Overseas', 'Indian capped'], 'Indian uncapped')

scores = {}
for f, key in [('batter', 'bat'), ('bowler', 'bowl')]:
    R = pd.read_csv(OUT / f'{f}_ratings.csv', index_col=0)
    scores[key] = R.consensus.dropna().round(3).to_dict()

category overrides (CSV said Indian, T20I nation is not India): 1


## Counts per player. Derived stats (SR, average, economy, %) are computed in the page.
batter row = [balls, runs, outs, fours, sixes, dots, innings]; bowler row = [legal balls, runs, wickets, fours, sixes, dots, extras]

In [6]:
PH = ['PP', 'MID', 'DEATH']


def bat_counts(d, extra_inns=True):
    d = d.assign(o=d.batter_out.astype(int), dt=d.bdot.astype(int), f4=d.is_four.astype(int), f6=d.is_six.astype(int))
    g = d.groupby('striker_id').agg(b=('runs', 'size'), r=('runs', 'sum'), o=('o', 'sum'), f4=('f4', 'sum'), f6=('f6', 'sum'), dt=('dt', 'sum'))
    g['i'] = d[['striker_id', 'match_id', 'innings']].drop_duplicates().groupby('striker_id').size()
    return g.fillna(0).astype(int)


def bowl_counts(d):
    d = d.assign(b=d.legal.astype(int), w=d.bowler_wkt.astype(int), dt=d.bowl_dot.astype(int), f4=d.is_four.astype(int), f6=d.is_six.astype(int), ex=d.bowl_extra.astype(int))
    g = d.groupby('bowler_id').agg(b=('b', 'sum'), r=('bowler_runs', 'sum'), w=('w', 'sum'), f4=('f4', 'sum'), f6=('f6', 'sum'), dt=('dt', 'sum'), ex=('ex', 'sum'))
    return g.fillna(0).astype(int)


def table(counts_fn, df, pid, pcol):
    df = df[df.phase.isin(PH) | (df.phase == 'SO')]
    out = {'a': counts_fn(df), 'p': counts_fn(df[df[pcol] >= PRESSURE])}
    out['ph'] = [counts_fn(df[df.phase == p]) for p in PH]
    out['sn'] = {s: counts_fn(df[df.season.astype(str) == s]) for s in sorted(df.season.astype(str).unique())}
    return out


def build(role, counts_fn, df, pid, pcol):
    SC = {'all': df, 'ipl': df[df.comp_group == 'IPL'], 'intl': df[df.comp_group == 'T20I']}     # international = T20Is
    T = {k: table(counts_fn, d, pid, pcol) for k, d in SC.items()}
    cols = list(T['all']['a'].columns)
    row = lambda g, i: [int(x) for x in g.loc[i, cols]] if i in g.index else [0] * len(cols)
    pack = lambda t, i: {'a': row(t['a'], i), 'p': row(t['p'], i), 'ph': [row(g, i) for g in t['ph']],
                         'sn': {s: row(g, i) for s, g in t['sn'].items() if i in g.index}}
    rows = []
    for i in E.index:
        if i not in T['all']['a'].index:
            continue
        e = E.loc[i]
        rows.append({'id': int(i), 'n': str(df_names.get(i, e['name'])), 'nat': str(e.nation).title().replace(' Of ', ' of ') if e.nation == e.nation else '',
                     'cat': e['category'], 'ord': e['orders'], 'ipl': e.current_ipl_team, 'sc': scores[role].get(i), 's': {k: pack(T[k], i) for k in T}})
    return rows


bp = pd.read_parquet(OUT / 'balls_pressure.parquet', columns=['striker_id', 'striker_name', 'bowler_id', 'bowler_name'])
df_names = pd.concat([bp[['striker_id', 'striker_name']].set_axis(['i', 'n'], axis=1), bp[['bowler_id', 'bowler_name']].set_axis(['i', 'n'], axis=1)]).drop_duplicates('i').set_index('i').n.to_dict()

payload = {'asof': pd.Timestamp.now().strftime('%d %b %Y'), 'file': ELIG.name,
           'bat': build('bat', bat_counts, Bt, 'striker_id', 'p_bat'), 'bowl': build('bowl', bowl_counts, Bw, 'bowler_id', 'p_bowl')}
html = (ROOT / 'dashboard' / 'template.html').read_text(encoding='utf-8').replace('__DATA__', json.dumps(payload, separators=(',', ':'), ensure_ascii=False))
(OUT / 'dashboard.html').write_text(html, encoding='utf-8')
print('players: batters %d, bowlers %d | available: %d / %d | file: %s | html %.0f KB' % (
    len(payload['bat']), len(payload['bowl']), sum(r['ipl'] == 'available' for r in payload['bat']), sum(r['ipl'] == 'available' for r in payload['bowl']), ELIG.name, len(html) / 1024))

players: batters 753, bowlers 554 | available: 540 / 385 | file: player_eligibility_1.csv | html 835 KB
